# 08_attrition_imputation
Two survivorship / missingness robustness checks for axis 2 (HTN primary):
(A1) Inverse-probability-of-censoring weighting (IPCW). We model the probability
that a person present at t0 is still observed at t1, then combine censoring
weights with the treatment IPW so the effect estimate is corrected for
non-random panel attrition.
(A2) Multiple imputation of missing baseline actionable/covariate values
(m=10, chained equations) with Rubin-pooled treatment effects, to show the
complete-case axis-2 estimate is not an artefact of missingness.

In [1]:
%run 00_config.ipynb

PROJ_DIR: /home/claude/recourse_khp
1y pairs: [(2019, 2020), (2020, 2021), (2021, 2022), (2022, 2023), (2023, 2024)]
2y pairs: [(2019, 2021), (2020, 2022), (2021, 2023), (2022, 2024)]
registry loaded
helpers loaded
00_config ready


In [2]:
import statsmodels.api as sm, statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
rng=np.random.default_rng(42)
panel=pd.read_parquet(os.path.join(DATA_DIR,"panel_long.parquet"))
tr=pd.read_parquet(os.path.join(DATA_DIR,"transitions_1y.parquet"))
tr["d_BMI"]=tr["BMI_t1"]-tr["BMI_t0"]

In [3]:
# --- A1: IPCW for panel attrition ---
# Universe: adults HTN-free at t0 in each wave-pair, regardless of whether they
# were re-observed at t1. Model observation (R=1) at t1 from t0 characteristics.
def baseline_universe():
    rows=[]
    seq=list(WAVES.keys())
    for w0,w1 in zip(seq[:-1],seq[1:]):
        a=panel[panel.wave==w0].copy()
        a=a[(a["HTN_dx"]==0)]  # at-risk baseline
        b=set(panel[panel.wave==w1][KEY])
        a["R"]=a[KEY].isin(b).astype(int)
        a["pair"]=f"{WAVES[w0]}_{WAVES[w1]}"; a["year0"]=WAVES[w0]
        rows.append(a)
    return pd.concat(rows,ignore_index=True)
U=baseline_universe()
Uc=U.dropna(subset=["BMI","age","SEX","H_INC_TOT"]).copy()
Uc["male"]=(Uc["SEX"]=="M").astype(int); Uc["inc0"]=Uc["H_INC_TOT"].fillna(Uc["H_INC_TOT"].median())
Xc=Uc[["BMI","age","male","inc0"]]
pc=Pipeline([("sc",StandardScaler()),("lr",LogisticRegression(max_iter=1000))]).fit(Xc,Uc["R"]).predict_proba(Xc)[:,1]
pc=np.clip(pc,0.05,0.999); Uc["cw"]=1/pc
print("attrition: overall re-observation rate =",round(Uc["R"].mean(),3))
print("censoring weight summary:",np.round(Uc["cw"].describe().values,3))
# merge censoring weight onto observed transitions by person+baseline year
cwmap=Uc[[KEY,"year0","cw"]]

attrition: overall re-observation rate = 0.901
censoring weight summary: [3.3329e+04 1.1110e+00 4.3000e-02 1.0330e+00 1.0760e+00 1.1020e+00
 1.1370e+00 1.2670e+00]


In [4]:
# Build HTN trial WITH censoring weights combined
def build_trial(df,target,tau=None):
    d=df[df[f"{target}_atrisk"]==1].dropna(subset=["BMI_t0","BMI_t1",f"{target}_onset","age_t0","SEX_t0","H_INC_TOT_t0"]).copy()
    red=-(d["BMI_t1"]-d["BMI_t0"])
    if tau is None: tau=float(red[red>0].quantile(0.75))
    d["TREAT"]=(red>=tau).astype(int); d["male"]=(d["SEX_t0"]=="M").astype(int)
    d["onset"]=d[f"{target}_onset"].astype(int); d["bmi0"]=d["BMI_t0"]; d["age0"]=d["age_t0"]
    d["inc0"]=d["H_INC_TOT_t0"].fillna(d["H_INC_TOT_t0"].median()); d["year0"]=d["year0"].astype("category")
    return d,tau
def ipw(d,cols):
    Xp=d[cols].copy()
    for c in cols: Xp[c]=Xp[c].fillna(Xp[c].median())
    e=Pipeline([("sc",StandardScaler()),("lr",LogisticRegression(max_iter=1000))]).fit(Xp,d["TREAT"]).predict_proba(Xp)[:,1]
    e=np.clip(e,0.02,0.98); pt=d["TREAT"].mean()
    return np.where(d["TREAT"]==1,pt/e,(1-pt)/(1-e))

H,tauH=build_trial(tr,"HTN")
H["year0_num"]=H["year0"].astype(int) if H["year0"].dtype.name!="category" else H["year0"].astype(str).astype(int) if False else H["year0"]
swH=ipw(H,["bmi0","age0","male","inc0"])
H=H.merge(cwmap,on=[KEY,"year0"],how="left")
H["cw"]=H["cw"].fillna(1.0)
def fit2(d,w):
    m=smf.glm("onset ~ TREAT + bmi0 + age0 + male + C(year0)",data=d,
              family=sm.families.Binomial(),freq_weights=w).fit(cov_type="HC1")
    return np.exp(m.params["TREAT"]),np.exp(m.conf_int().loc["TREAT"]).values
or_ipw,ci_ipw=fit2(H,swH)
or_comb,ci_comb=fit2(H,swH*H["cw"].values)
a1=pd.DataFrame([
 {"weighting":"treatment IPW only","OR":round(or_ipw,3),"lo":round(ci_ipw[0],3),"hi":round(ci_ipw[1],3)},
 {"weighting":"IPW x IPCW (attrition-corrected)","OR":round(or_comb,3),"lo":round(ci_comb[0],3),"hi":round(ci_comb[1],3)},
])
savetable(a1,"t08_a1_ipcw", index=False)
print(a1.to_string(index=False))

saved: t08_a1_ipcw.csv
                       weighting    OR    lo    hi
              treatment IPW only 0.608 0.450 0.822
IPW x IPCW (attrition-corrected) 0.609 0.456 0.813


In [5]:
# --- A2: multiple imputation of baseline actionable/covariates (m=10) ---
# Impute at the transition level for the HTN at-risk universe, keeping onset observed.
cols_imp=["BMI_t0","age_t0","H_INC_TOT_t0","PA_WALK_t0","ALC_FREQ_t0"]
base=tr[tr["HTN_atrisk"]==1].copy()
base["male"]=(base["SEX_t0"]=="M").astype(int)
base["onset"]=base["HTN_onset"].astype(int)
base["year0"]=base["year0"].astype(int)
present=[c for c in cols_imp if c in base.columns]
core=base[present+["BMI_t1","male","onset","year0"]].copy()

m_imp=10; ests=[]; ses=[]
for k in range(m_imp):
    imp=IterativeImputer(max_iter=10,sample_posterior=True,random_state=k)
    Z=pd.DataFrame(imp.fit_transform(core[present+["BMI_t1"]]),columns=present+["BMI_t1"])
    d=core.copy()
    for c in present+["BMI_t1"]: d[c]=Z[c].values
    red=-(d["BMI_t1"]-d["BMI_t0"]); tau=float(red[red>0].quantile(0.75))
    d["TREAT"]=(red>=tau).astype(int); d["bmi0"]=d["BMI_t0"]; d["age0"]=d["age_t0"]
    m=smf.glm("onset ~ TREAT + bmi0 + age0 + male + C(year0)",data=d,
              family=sm.families.Binomial()).fit()
    ests.append(m.params["TREAT"]); ses.append(m.bse["TREAT"])
ests=np.array(ests); ses=np.array(ses)
# Rubin's rules
qbar=ests.mean(); Ubar=(ses**2).mean(); Bv=ests.var(ddof=1)
Tv=Ubar+(1+1/m_imp)*Bv; se_pool=np.sqrt(Tv)
or_pool=np.exp(qbar); lo=np.exp(qbar-1.96*se_pool); hi=np.exp(qbar+1.96*se_pool)
a2=pd.DataFrame([{"method":"MI (m=10), Rubin-pooled","OR":round(or_pool,3),
                  "lo":round(lo,3),"hi":round(hi,3),"fmi_B_over_T":round((1+1/m_imp)*Bv/Tv,3)}])
savetable(a2,"t08_a2_multiple_imputation", index=False)
print(a2.to_string(index=False))

saved: t08_a2_multiple_imputation.csv
                 method    OR    lo    hi  fmi_B_over_T
MI (m=10), Rubin-pooled 0.677 0.514 0.892         0.011


In [6]:
# Figure: axis-2 HTN OR under each robustness treatment (complete-case, IPCW, MI)
cc_or,cc_ci=or_ipw,ci_ipw
pts=[("Complete-case IPW",cc_or,cc_ci[0],cc_ci[1]),
     ("+ IPCW attrition",or_comb,ci_comb[0],ci_comb[1]),
     ("Multiple imputation",or_pool,lo,hi)]
fig,ax=plt.subplots(figsize=(6.0,3.6))
yy=np.arange(len(pts))
for i,(lab,o,l,h) in enumerate(pts):
    ax.plot([l,h],[i,i],color="#333333",lw=1.4)
    ax.plot(o,i,"o",color="#000000",ms=5)
ax.axvline(1.0,color="#999999",lw=0.8,ls="--")
ax.set_yticks(yy); ax.set_yticklabels([p[0] for p in pts])
ax.set_xlabel("HTN onset OR (treated vs control)")
savefig(fig,"f08_axis2_robustness"); plt.close(fig)
print("robustness figure saved")

saved: f08_axis2_robustness.png / f08_axis2_robustness.pdf
robustness figure saved
